# Feather v1 -- Offline Inference

Load the packaged weights (npz inside the uploaded `feather-v1-kaggle.tar.gz`),
tokenize a prompt with the byte tokenizer, generate, decode. Works fully offline.

In [ ]:
import json
import numpy as np
from feather_v1 import FeatherV1Model, FeatherV1Config
from feather_v1.hardware import kaggle_env, summary
from feather_v1.utils import byte_tokenize, byte_decode

env = kaggle_env()
print("is_kaggle:", env["is_kaggle"], "| run_type:", env["kernel_run_type"])
print("ram_gb:", env["ram_gb"], "| internet:", env["has_internet"], "| cuda:", env["cuda_devices"])

In [ ]:
# locate weights: Kaggle input first, then local build
def load_weights():
    import pathlib as pl
    for cand in (
        '/kaggle/input/feather-v1-model/feather-v1-kaggle/feather-v1-kaggle.pt',
        'models/kaggle/feather-v1-kaggle.pt',
    ):
        if pl.Path(cand).is_file():
            return FeatherV1Model.from_weights(cand)
    raise FileNotFoundError('no feather-v1-kaggle.pt: run kaggle_dataset.py first')
model = load_weights()
print('loaded: dim', model.config.dim, 'vocab', model.config.vocab_size)

In [ ]:
# offline proof: block outbound sockets and re-check
import socket
def blocked(*a, **k): raise OSError('offline sandbox')
socket.socket.connect = blocked
socket.create_connection = blocked
from feather_v1.hardware import has_internet
print('internet reachable after blocking:', has_internet(timeout=0.05))

In [ ]:
# byte-tokenize prompt -> one-hot rows -> generate -> decode
def prompt_rows(text, dim):
    ids = byte_tokenize(text)
    prompt = np.zeros((ids.size, dim))
    for i, tok in enumerate(ids):
        prompt[i, int(tok) % dim] = 1.0
    return prompt

prompt_text = 'Feather v1 runs CPU-only on Kaggle.'
prompt = prompt_rows(prompt_text, model.config.dim)
print('prompt byte ids:', byte_tokenize(prompt_text)[:24], '...')
seq = model.generate(prompt, steps=8)
print('generated last-row token id:', int(seq[-1].argmax()))
print('round-trip byte decode:', byte_decode(byte_tokenize(prompt_text))[:32])

The archive is a Kaggle **Dataset** input (Upload > New dataset);
notebooks then use `/kaggle/input/feather-v1-model/...` unchanged.